In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import torch_geometric
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data
import numpy as np
import os
import random
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
import pandas as pd
import matplotlib.pyplot as plt
from CodecManager import CodecManager
import optuna 
from torch.optim.lr_scheduler import StepLR

seed_value = 42
torch.manual_seed(seed_value)
torch.cuda.manual_seed(seed_value)
torch.cuda.manual_seed_all(seed_value)
np.random.seed(seed_value)
random.seed(seed_value)

In [ ]:
print(f"PyTorch version: {torch.__version__}")
print(f"PyTorch Geometric version: {torch_geometric.__version__}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.cuda.empty_cache()

# Define the model directory and filename
model_dir = 'Model_architecture\\model_v1\\'
model_filename = "best_model.pth"

# Load mesh coordinates and adjacency matrix
coordinates = np.array(pd.read_csv('Dimensionality_reduction\Adjency_matrix\\coordinates.csv', header = None))
adjacency_matrix = pd.read_csv('Dimensionality_reduction\Adjency_matrix\\adjacency_matrix.csv')

# Define the study name and storage URL for Optuna
study_name = 'autoencoder_GCN_optimization_1'
storage_url = 'sqlite:///autoencoder_GCN_optimization_1.db'  # Change to your preferred storage URL

# Load dataset
pressure_dataset = np.load('Dataset\\dataset.npy')
# Load grid data
grid_data = pd.read_csv('Dataset\\grid_data.dat', sep= ' ')


# Set to True to shuffle the data before training
shuffle_data = False

# Transfer Learning: True-> loads the stored weights and trains from there; False-> trains from new
Transfer_Learning_flag = True
# Training Flag: True-> model.fit is executed and saved; False-> not trained neither stored
Execute_Train_flag = False

# Check if the directory already exists
if not os.path.exists(model_dir):
    # If it doesn't exist, create it
    os.makedirs(model_dir)

In [ ]:
# Define the dataset 
print( '-> Shape of the loaded matrix: ',pressure_dataset.shape) # Shape: (samples, nodes, features)~(70, 86840, 9), with features being [x, y, z, CP, CF_x, CF_y, CF_z, M, AoA]

# Select the features for input and output
X = pressure_dataset[:,:,[0,1,2,7,8]] # Input
# Y = np.expand_dims(pressure_dataset[:,:,3],axis=2)         # Output
Y = pressure_dataset[:,:,3:7]         # Output
print( '-> Shape of the input matrix: ',X.shape)
print( '-> Shape of the output matrix: ',Y.shape)

In [ ]:
edge_indices_orig = adjacency_matrix[['point_i', 'point_j']]
edge_distances = adjacency_matrix[['distance']]
point_id_orig = coordinates[:,-1].astype(int)

space1 = CodecManager(
    npoints=28600, 
    point_id_orig = np.argsort(point_id_orig), 
    edge_indices_orig = edge_indices_orig, 
    edge_distances = edge_distances
)

space2 = CodecManager(
    npoints=9600, 
    point_id_orig = np.argsort(space1.point_id_reduced_orig), 
    edge_indices_orig = space1.edge_indices_reduced,
    edge_distances = space1.edge_distances_reduced
)

encoder_sparse_interpolation_1, decoder_sparse_interpolation_1 = space1.get_interpolation_matrix()
encoder_sparse_interpolation_2, decoder_sparse_interpolation_2 = space2.get_interpolation_matrix()

In [ ]:
## Feature normalisation:
scaler = MinMaxScaler(feature_range=(-1, 1))
samples, points, variables = X.shape
X = np.reshape(X, newshape=(-1, variables))
X = scaler.fit_transform(X)
X = np.reshape(X, newshape=(samples, points, variables))

scalery = MinMaxScaler(feature_range=(-1, 1))
samples, points, variables = Y.shape
Y = np.reshape(Y, newshape=(-1, variables))
Y = scalery.fit_transform(Y)
Y = np.reshape(Y, newshape=(samples, points, variables))

In [ ]:
## Split data into training and test sets
train_ratio = 0.58 
test_ratio = 0.20
validation_ratio = 0.20

X_train, X_val, Y_train, Y_val = train_test_split(X, Y, test_size=1 - train_ratio , shuffle=shuffle_data)
X_test, X_val, Y_test, Y_val = train_test_split(X_val, Y_val, test_size=test_ratio/(test_ratio + validation_ratio) , shuffle=shuffle_data)


print("Train Input shape:", X_train.shape)
print("Test Input shape:", X_test.shape)
print("Validation Input shape:", X_val.shape)
print("Train Output shape:", Y_train.shape)
print("Test Output shape:", Y_test.shape)
print("Validation Output shape:", Y_val.shape)


In [ ]:
# Define a custom PyTorch Geometric Dataset
class CustomGraphDataset(Dataset):
    """
    Custom dataset for graph data compatible with PyTorch Geometric.
    Stores input features X and target labels Y.
    """
    def __init__(self, X, Y):
        """
        Args:
            X (np.ndarray): Input features of shape (samples, nodes, features)
            Y (np.ndarray): Target labels of shape (samples, nodes, features)
        """
        self.X = X
        self.Y = Y

    def __len__(self):
        """Returns the number of samples in the dataset."""
        return len(self.X)

    def __getitem__(self, idx):
        """
        Retrieves the sample at the given index as a PyTorch Geometric Data object.
        Args:
            idx (int): Index of the sample
        Returns:
            Data: PyTorch Geometric Data object with x and y attributes
        """
        x = torch.tensor(self.X[idx], dtype=torch.float32)  # Convert input to tensor
        y = torch.tensor(self.Y[idx], dtype=torch.float32)  # Convert target to tensor
        data = Data(x=x, y=y)  # Create Data object
        return data

# Create dataset instances for training, validation, and testing
train_dataset = CustomGraphDataset(X_train, Y_train)  # Training dataset
val_dataset = CustomGraphDataset(X_val, Y_val)        # Validation dataset
test_dataset = CustomGraphDataset(X_test, Y_test)     # Test dataset

def collate(data_list):
    """
    Custom collate function to move data to the appropriate device.
    Args:
        data_list (list): List of Data objects
    Returns:
        list: List of Data objects moved to device if CUDA is available
    """
    if torch.cuda.is_available():
        data_list = [data.to(device) for data in data_list]  # Move to GPU if available
    return data_list

batch_size = 1  # Set batch size for DataLoader

def seed_worker(worker_id):
    """
    Seeds the random number generators for reproducibility in DataLoader workers.
    Args:
        worker_id (int): Worker ID
    """
    worker_seed = torch.initial_seed() % 2**32  # Get worker seed
    np.random.seed(worker_seed)                 # Seed numpy RNG
    random.seed(worker_seed)                    # Seed python RNG

g = torch.Generator()           # Create a torch generator
g.manual_seed(seed_value)       # Set manual seed for reproducibility

# Create DataLoaders for train, validation, and test sets
train_loader = DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True,
    worker_init_fn=seed_worker, generator=g, collate_fn=collate
)  # Training DataLoader

val_loader = DataLoader(
    val_dataset, batch_size=batch_size, shuffle=True,
    worker_init_fn=seed_worker, generator=g, collate_fn=collate
)  # Validation DataLoader

test_loader = DataLoader(
    test_dataset, batch_size=batch_size, shuffle=False,
    worker_init_fn=seed_worker, generator=g, collate_fn=collate
)  # Test DataLoader

# Print DataLoader statistics
total_samples = len(train_loader.dataset)  # Total samples in train set
batch_size = train_loader.batch_size       # Batch size used
num_batches = len(train_loader)            # Number of batches

print(f"Total number of samples in train_loader: {total_samples}")
print(f"Batch size in train_loader: {batch_size}")
print(f"Number of batches in train_loader: {num_batches}")

batch = next(iter(train_loader))  # Get a batch from the train_loader
print("Batch:", batch)

In [ ]:
import torch
from torch_geometric.nn import GCNConv

import torch.nn as nn

class GraphConv(torch.nn.Module):
    """
    Graph convolutional layer using GCNConv with optional output activation.
    """
    def __init__(self, in_channels, out_channels, edge_indices, edge_distances, output=False):
        super(GraphConv, self).__init__()
        self.edge_indices = edge_indices  # Edge indices for the graph
        self.edge_distances = edge_distances  # Edge weights/distances
        self.in_channels = in_channels  # Number of input features
        self.out_channels = out_channels  # Number of output features
        self.conv = GCNConv(in_channels, out_channels, add_self_loops=True, improved=True, cached=True)  # GCN layer
        self.prelu = nn.PReLU()  # PReLU activation
        self.output = output  # Whether this is an output layer

    def forward(self, x):
        """
        Forward pass for the graph convolutional layer.
        Args:
            x (Tensor): Input node features.
        Returns:
            Tensor: Output node features.
        """
        x = self.conv(x, edge_index=self.edge_indices, edge_weight=self.edge_distances)  # Apply GCN
        if not self.output:
            x = self.prelu(x)  # Apply activation if not output
        return x

class Encoder(torch.nn.Module):
    """
    Encoder module for pooling/encoding node features using a sparse interpolation matrix.
    """
    def __init__(self, edge_indices_reduced, edge_distances_reduced, point_id_index_reduced, encoder_interpolation_matrix):
        super(Encoder, self).__init__()
        self.edge_indices_reduced = edge_indices_reduced  # Reduced edge indices
        self.edge_distances_reduced = edge_distances_reduced  # Reduced edge distances
        self.point_id_index_reduced = point_id_index_reduced  # Reduced point indices
        self.encoder_interpolation_matrix = encoder_interpolation_matrix  # Sparse interpolation matrix

    def forward(self, x):
        """
        Forward pass for the encoder.
        Args:
            x (Tensor): Input node features.
        Returns:
            Tensor: Encoded node features.
        """
        x = torch.sparse.mm(self.encoder_interpolation_matrix, x)  # Pool features using interpolation matrix
        return x

class Decoder(torch.nn.Module):
    """
    Decoder module for unpooling/decoding node features using a sparse interpolation matrix.
    """
    def __init__(self, decoder_interpolation_matrix):
        super(Decoder, self).__init__()
        self.decoder_interpolation_matrix = decoder_interpolation_matrix  # Sparse interpolation matrix

    def unpool(self, x_encoded):
        """
        Unpool encoded features to higher resolution.
        Args:
            x_encoded (Tensor): Encoded node features.
        Returns:
            Tensor: Decoded node features.
        """
        x = torch.sparse.mm(self.decoder_interpolation_matrix, x_encoded)  # Unpool features
        return x

    def forward(self, x):
        """
        Forward pass for the decoder.
        Args:
            x (Tensor): Encoded node features.
        Returns:
            Tensor: Decoded node features.
        """
        return self.unpool(x)

class IterableGcn(nn.Module):
    """
    Sequential container for a list of GCN layers.
    """
    def __init__(self, layer_list):
        super(IterableGcn, self).__init__()
        self.layers = nn.ModuleList(layer_list)  # Store layers in a ModuleList

    def forward(self, x):
        """
        Forward pass through all layers in sequence.
        Args:
            x (Tensor): Input node features.
        Returns:
            Tensor: Output node features.
        """
        for layer in self.layers:
            x = layer(x)  # Pass through each layer
        return x

class GraphAutoencoder(torch.nn.Module):
    """
    Hierarchical Graph Autoencoder with multiple pooling and unpooling stages.
    """
    def __init__(self,
                 in_channels,
                 edge_indices, 
                 edge_distances, 
                 edge_indices_reduced, 
                 edge_distances_reduced, 
                 encoder_interpolation_matrix,
                 decoder_interpolation_matrix,
                 point_id_index_reduced,
                 edge_indices_reduced_1, 
                 edge_distances_reduced_1, 
                 encoder_interpolation_matrix_1,
                 decoder_interpolation_matrix_1,
                 point_id_index_reduced_1,
                 n_units_input,
                 n_units_output,
                 n_layers_orig,
                 n_units_orig,
                 n_layers_pool_1,
                 n_units_pool_1,
                 n_layers_pool_2,
                 n_units_pool_2
                 ):
        super(GraphAutoencoder, self).__init__()

        # Store graph and pooling/unpooling parameters
        self.edge_indices = edge_indices
        self.edge_distances = edge_distances
        self.edge_indices_reduced = edge_indices_reduced
        self.edge_distances_reduced = edge_distances_reduced
        self.encoder_interpolation_matrix = encoder_interpolation_matrix
        self.decoder_interpolation_matrix = decoder_interpolation_matrix
        self.point_id_index_reduced = point_id_index_reduced
        self.edge_indices_reduced_1 = edge_indices_reduced_1
        self.edge_distances_reduced_1 = edge_distances_reduced_1
        self.encoder_interpolation_matrix_1 = encoder_interpolation_matrix_1
        self.decoder_interpolation_matrix_1 = decoder_interpolation_matrix_1
        self.point_id_index_reduced_1 = point_id_index_reduced_1

        self.n_units_input = n_units_input
        self.n_units_output = n_units_output
        self.n_layers_orig = n_layers_orig
        self.n_units_orig = n_units_orig
        self.n_layers_pool_1 = n_layers_pool_1
        self.n_units_pool_1 = n_units_pool_1
        self.n_layers_pool_2 = n_layers_pool_2
        self.n_units_pool_2 = n_units_pool_2

        # Encoder: initial graph convolution
        self.graphconv_input = GraphConv(in_channels, n_units_input, edge_indices, edge_distances).to(device)

        # Encoder: original space GCN layers
        self.graphconv_enc_orig = []
        self.graphconv_dec_orig = []
        for i in range(n_layers_orig):
            if i == 0:
                self.graphconv_enc_orig.append(GraphConv(n_units_input, n_units_orig[0], edge_indices, edge_distances).to(device))
                self.graphconv_dec_orig.append(GraphConv(n_units_orig[0], n_units_output, edge_indices, edge_distances).to(device))
            else:
                self.graphconv_enc_orig.append(GraphConv(n_units_orig[i-1], n_units_orig[i], edge_indices, edge_distances).to(device))
                self.graphconv_dec_orig.append(GraphConv(n_units_orig[i], n_units_orig[i-1], edge_indices, edge_distances).to(device))

        # Encoder: first pooling stage
        self.encode = Encoder(edge_indices_reduced, edge_distances_reduced, point_id_index_reduced, encoder_interpolation_matrix).to(device)
        self.graphconv_pool_1 = []
        self.graphconv_unpool_1 = []
        for i in range(n_layers_pool_1):
            if i == 0:
                self.graphconv_pool_1.append(GraphConv(n_units_orig[-1], n_units_pool_1[0], edge_indices_reduced, edge_distances_reduced).to(device))
                self.graphconv_unpool_1.append(GraphConv(n_units_pool_1[0], n_units_orig[-1], edge_indices_reduced, edge_distances_reduced).to(device))
            else:
                self.graphconv_pool_1.append(GraphConv(n_units_pool_1[i-1], n_units_pool_1[i], edge_indices_reduced, edge_distances_reduced).to(device))
                self.graphconv_unpool_1.append(GraphConv(n_units_pool_1[i], n_units_pool_1[i-1], edge_indices_reduced, edge_distances_reduced).to(device))

        # Encoder: second pooling stage
        self.encode_1 = Encoder(edge_indices_reduced_1, edge_distances_reduced_1, point_id_index_reduced_1, encoder_interpolation_matrix_1).to(device)
        self.graphconv_pool_2 = []
        self.graphconv_unpool_2 = []
        for i in range(n_layers_pool_2):
            if i == 0:
                self.graphconv_pool_2.append(GraphConv(n_units_pool_1[-1], n_units_pool_2[0], edge_indices_reduced_1, edge_distances_reduced_1).to(device))
                self.graphconv_unpool_2.append(GraphConv(n_units_pool_2[0], n_units_pool_1[-1], edge_indices_reduced_1, edge_distances_reduced_1).to(device))
            else:
                self.graphconv_pool_2.append(GraphConv(n_units_pool_2[i-1], n_units_pool_2[i], edge_indices_reduced_1, edge_distances_reduced_1).to(device))
                self.graphconv_unpool_2.append(GraphConv(n_units_pool_2[i], n_units_pool_2[i-1], edge_indices_reduced_1, edge_distances_reduced_1).to(device))

        # Decoder: wrap unpooling and decoding layers in IterableGcn for sequential application
        self.graphconv_unpool_2 = IterableGcn(list(reversed(self.graphconv_unpool_2))).to(device)
        self.decode_1 = Decoder(decoder_interpolation_matrix_1).to(device)
        self.graphconv_unpool_1 = IterableGcn(list(reversed(self.graphconv_unpool_1))).to(device)
        self.decode = Decoder(decoder_interpolation_matrix).to(device)
        self.graphconv_dec_orig = IterableGcn(list(reversed(self.graphconv_dec_orig))).to(device)

        # Wrap encoder GCNs in IterableGcn
        self.graphconv_enc_orig = IterableGcn(self.graphconv_enc_orig).to(device)
        self.graphconv_pool_1 = IterableGcn(self.graphconv_pool_1).to(device)
        self.graphconv_pool_2 = IterableGcn(self.graphconv_pool_2).to(device)

        # Output layers for different variables
        variables_to_predict = 1
        self.out_conv_cp = GraphConv(n_units_output, variables_to_predict, edge_indices, edge_distances, output=True).to(device)
        self.out_conv_cfx = GraphConv(n_units_output, variables_to_predict, edge_indices, edge_distances, output=True).to(device)
        self.out_conv_cfy = GraphConv(n_units_output, variables_to_predict, edge_indices, edge_distances, output=True).to(device)
        self.out_conv_cfz = GraphConv(n_units_output, variables_to_predict, edge_indices, edge_distances, output=True).to(device)

        # Freeze encoder and decoder parameters
        self.set_requires_grad(False, [self.encode, self.decode])

    def set_requires_grad(self, requires_grad, modules):
        """
        Set requires_grad for all parameters in the given modules.
        Args:
            requires_grad (bool): Whether gradients are required.
            modules (list): List of modules.
        """
        for module in modules:
            for param in module.parameters():
                param.requires_grad = requires_grad

    def forward(self, x):
        """
        Forward pass for the hierarchical graph autoencoder.
        Args:
            x (Tensor): Input node features.
        Returns:
            Tensor: Output predictions.
        """
        x = x.to(device)  # Move input to device

        # Input GCN
        x = self.graphconv_input(x)

        # Original space GCNs
        x = self.graphconv_enc_orig(x)

        # First pooling stage
        x_enc = self.encode(x)
        x_enc = self.graphconv_pool_1(x_enc)

        # Second pooling stage
        x_enc_1 = self.encode_1(x_enc)
        x_enc_1 = self.graphconv_pool_2(x_enc_1)

        # Decoder: unpool and decode
        x_enc_1 = self.graphconv_unpool_2(x_enc_1)
        x_dec_1 = self.decode_1(x_enc_1)

        x_dec_1 = self.graphconv_unpool_1(x_dec_1)
        x_dec = self.decode(x_dec_1)

        # Original space decoder GCNs
        x_dec = self.graphconv_dec_orig(x_dec)

        # Output layers for each variable
        x_cp = self.out_conv_cp(x_dec)
        x_cfx = self.out_conv_cfx(x_dec)
        x_cfy = self.out_conv_cfy(x_dec)
        x_cfz = self.out_conv_cfz(x_dec)

        # Concatenate outputs
        x_out = torch.cat((x_cp, x_cfx, x_cfy, x_cfz), dim=1)

        return x_out


In [ ]:
# study_name = 'autoencoder_GCN_optimization_1'
# storage_url = 'sqlite:///autoencoder_GCN_optimization_1.db'  # Change to your preferred storage URL
# loaded_study = optuna.delete_study(study_name=study_name, storage=storage_url)


In [ ]:
# Compute Aero Coeff for Loss

# Convert scaler min/max to torch tensors for denormalization, move to device
X_min = torch.tensor(scaler.data_min_, dtype=torch.float32).to(device)
X_max = torch.tensor(scaler.data_max_, dtype=torch.float32).to(device)
Y_min = torch.tensor(scalery.data_min_, dtype=torch.float32).to(device)
Y_max = torch.tensor(scalery.data_max_, dtype=torch.float32).to(device)

# Convert grid and coordinate data to torch tensors, move to device
cell_area = torch.tensor(grid_data.Cell_Volume, dtype=torch.float32).to(device)
X_Grid_K_Unit_Normal = torch.tensor(grid_data.X_Grid_K_Unit_Normal, dtype=torch.float32).to(device)
Y_Grid_K_Unit_Normal = torch.tensor(grid_data.Y_Grid_K_Unit_Normal, dtype=torch.float32).to(device)
Z_Grid_K_Unit_Normal = torch.tensor(grid_data.Z_Grid_K_Unit_Normal, dtype=torch.float32).to(device)
coordinates_tensor = torch.tensor(coordinates, dtype=torch.float32).to(device)

def compute_aero_coeff(ds, AoA):
    """
    Compute aerodynamic moment coefficient for a given dataset and angle of attack.

    Args:
        ds (Tensor): Data tensor with shape (num_points, features).
        AoA (float): Angle of attack in degrees.

    Returns:
        Tensor: Scalar moment coefficient.
    """
    global X_Grid_K_Unit_Normal, Y_Grid_K_Unit_Normal, Z_Grid_K_Unit_Normal, cell_area, coordinates_tensor

    xref = 0.12192  # Reference x location
    chord = 0.4064  # Chord length
    span = chord * 2  # Span length
    area = span * chord  # Reference area
    aoa = AoA * np.pi / 180.0  # Convert AoA to radians

    shear_x = ds[:, 1]  # Extract shear force in x-direction
    shear_z = ds[:, 3]  # Extract shear force in z-direction
    cp = ds[:, 0]       # Extract pressure coefficient

    # Calculate surface tractions
    taux = (shear_x - X_Grid_K_Unit_Normal * cp) / area
    tauz = (shear_z - Z_Grid_K_Unit_Normal * cp) / area

    # Calculate moment coefficient (about y-axis)
    my = (
        coordinates_tensor[:, 2] * (taux * torch.cos(aoa) + tauz * torch.sin(aoa))
        - (coordinates_tensor[:, 0] - xref) * (tauz * torch.cos(aoa) - taux * torch.sin(aoa))
    ) / chord

    moment = torch.sum(my * cell_area)  # Integrate over all cells

    return moment

def integral_load(x, y, out):
    """
    Compute the squared error between predicted and true aerodynamic moments.

    Args:
        x (Tensor): Input features (normalized).
        y (Tensor): True output (normalized).
        out (Tensor): Predicted output (normalized).

    Returns:
        Tensor: Mean squared error of the moment.
    """
    global X_min, X_max, Y_min, Y_max

    # Denormalize input and outputs
    x_den = x - X_min / (X_max - X_min)
    out_den = out - Y_min / (Y_max - Y_min)
    y_den = y - Y_min / (Y_max - Y_min)

    AoA = x_den[0, 4]  # Extract AoA from denormalized input

    # Compute aerodynamic moments for prediction and ground truth
    moment_NN = compute_aero_coeff(out_den, AoA)
    moment_CFD = compute_aero_coeff(y_den, AoA)

    squared_error = (moment_CFD - moment_NN) ** 2  # Squared error
    error_moment = torch.mean(squared_error)        # Mean squared error

    return error_moment


In [ ]:
def objective(trial):
    """
    Objective function for Optuna hyperparameter optimization.
    Builds and trains a GraphAutoencoder model with hyperparameters suggested by Optuna,
    and returns the validation loss for the trial.
    """
    # Suggest number of layers for each encoder section
    layers_orig = trial.suggest_int('layers_orig', 1, 3, step=1)  # Number of layers in original encoder
    layers_pool_1 = trial.suggest_int('layers_pool_1', 1, 3, step=1)  # Number of layers in first pooling encoder
    layers_pool_2 = trial.suggest_int('layers_pool_2', 1, 3, step=1)  # Number of layers in second pooling encoder

    # Suggest units for input and output layers
    n_units_input = trial.suggest_int('n_units_input', 32, 256, step=16)  # Units in input layer
    n_units_output = trial.suggest_int('n_units_output', 32, 256, step=16)  # Units in output layer

    # Suggest units for each layer in each encoder section
    units_orig = [trial.suggest_int(f'units_orig_{i+1}', 32, 256, step=16) for i in range(layers_orig)]  # Units in original encoder layers
    units_pool_1 = [trial.suggest_int(f'units_pool_1_{i+1}', 32, 384, step=16) for i in range(layers_pool_1)]  # Units in first pooling encoder layers
    units_pool_2 = [trial.suggest_int(f'units_pool_2_{i+1}', 32, 512, step=16) for i in range(layers_pool_2)]  # Units in second pooling encoder layers

    # Instantiate the GraphAutoencoder model with suggested hyperparameters
    model = GraphAutoencoder(
        in_channels=5,
        edge_indices=space1.get_edge_indices().to(device),
        edge_distances=space1.get_edge_distances().to(device),
        edge_indices_reduced=space1.get_indices_reduced().to(device),
        edge_distances_reduced=space1.get_edge_distances_reduced().to(device),
        encoder_interpolation_matrix=encoder_sparse_interpolation_1.to(device),
        decoder_interpolation_matrix=decoder_sparse_interpolation_1.to(device),
        point_id_index_reduced=space1.get_point_id_index_reduced().to(device),
        edge_indices_reduced_1=space2.get_indices_reduced().to(device),
        edge_distances_reduced_1=space2.get_edge_distances_reduced().to(device),
        encoder_interpolation_matrix_1=encoder_sparse_interpolation_2.to(device),
        decoder_interpolation_matrix_1=decoder_sparse_interpolation_2.to(device),
        point_id_index_reduced_1=space2.get_point_id_index_reduced().to(device),
        n_units_input=n_units_input,
        n_units_output=n_units_output,
        n_layers_orig=layers_orig,
        n_units_orig=units_orig,
        n_layers_pool_1=layers_pool_1,
        n_units_pool_1=units_pool_1,
        n_layers_pool_2=layers_pool_2,
        n_units_pool_2=units_pool_2
    ).to(device)  # Move model to device (CPU or GPU)

    n_epochs = 500  # Number of training epochs
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)  # Adam optimizer
    mse_loss = nn.MSELoss()  # Mean squared error loss

    # Learning rate scheduler to reduce LR every 30 epochs by a factor of 0.9
    scheduler = StepLR(optimizer, step_size=30, gamma=0.9)

    early_stopping_patience = 50  # Early stopping patience
    best_val_loss = float("inf")  # Initialize best validation loss
    epochs_without_improvement = 0  # Counter for early stopping
    lambda_factor = 0.01  # Weight for error_moment in loss

    for epoch in range(n_epochs):
        model.train()  # Set model to training mode
        total_loss = 0  # Accumulate training loss

        for data in train_loader:  # Iterate over training batches
            optimizer.zero_grad()  # Zero gradients
            for i in range(len(data)):  # Iterate over graphs in batch
                x, y = data[i].x, data[i].y  # Get input and target
                out = model(x)  # Forward pass
                loss = mse_loss(out, y)  # Compute MSE loss
                error_moment = integral_load(x, y, out)  # Compute additional error term
                combined_loss = loss + lambda_factor * error_moment  # Combine losses
                combined_loss.backward()  # Backpropagation
                optimizer.step()  # Update weights
                total_loss += combined_loss.item()  # Accumulate loss

        average_train_loss = total_loss / len(train_loader)  # Average training loss

        # Validation phase
        model.eval()  # Set model to evaluation mode
        total_val_loss = 0  # Accumulate validation loss
        with torch.no_grad():  # No gradient computation
            for data in val_loader:  # Iterate over validation batches
                for i in range(len(data)):  # Iterate over graphs in batch
                    x, y = data[i].x, data[i].y  # Get input and target
                    out = model(x)  # Forward pass
                    val_loss = mse_loss(out, y)  # Compute MSE loss
                    error_moment_val = integral_load(x, y, out)  # Compute additional error term
                    combined_loss_val = val_loss + lambda_factor * error_moment_val  # Combine losses
                    total_val_loss += combined_loss_val.item()  # Accumulate loss

        average_val_loss = total_val_loss / len(val_loader)  # Average validation loss

        current_lr = optimizer.param_groups[0]['lr']  # Get current learning rate
        if current_lr > 0.00005:
            scheduler.step()  # Step the learning rate scheduler

        # Early stopping check
        if average_val_loss < best_val_loss:
            best_val_loss = average_val_loss  # Update best validation loss
            epochs_without_improvement = 0  # Reset counter
        else:
            epochs_without_improvement += 1  # Increment counter

        # Check for early stopping
        if epochs_without_improvement >= early_stopping_patience:
            break  # Stop training if no improvement

        # Update trial values with the lowest loss for the current trial
        if 'lowest_loss' not in trial.user_attrs or average_val_loss < trial.user_attrs['lowest_loss']:
            trial.set_user_attr('lowest_loss', average_val_loss)  # Store lowest loss in trial attributes

    return average_val_loss  # Return validation loss for Optuna

trial_values = []  # List to store the lowest loss value for each optimizer trial

# Create a study object with TPE sampler (Bayesian optimization) and optimize hyperparameters
study = optuna.create_study(
    study_name=study_name,
    storage=storage_url,
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=seed_value)
)  # Create Optuna study

study.optimize(objective, n_trials=20)  # Run optimization for 20 trials

# Extract the lowest loss value for each trial from the study trials
for trial in study.trials:
    trial_values.append(trial.user_attrs['lowest_loss'])  # Append lowest loss to list


In [ ]:
# Store the loss value at each trail in a file
with open("val_loss_vs_optimizer_trials.txt", "w") as f:
    for trial_number, loss_value in enumerate(trial_values, start=1):
        f.write(f"Trial {trial_number}: {loss_value}\n")

In [ ]:
# Get the best hyperparameters
best_params = study.best_params
layers_orig = best_params['layers_orig']
layers_pool_1 = best_params['layers_pool_1']
layers_pool_2 = best_params['layers_pool_2']
n_units_input = best_params['n_units_input']
n_units_output = best_params['n_units_output']
units_orig =   [best_params[f'units_orig_{str(i+1)}'] for i in range(int(layers_orig))]
units_pool_1 = [best_params[f'units_pool_1_{str(i+1)}'] for i in range(int(layers_pool_1))]
units_pool_2 = [best_params[f'units_pool_2_{str(i+1)}'] for i in range(int(layers_pool_2))]
        
model = GraphAutoencoder(in_channels=5,
                edge_indices = space1.get_edge_indices().to(device), 
                edge_distances = space1.get_edge_distances().to(device), 
                edge_indices_reduced = space1.get_indices_reduced().to(device), 
                edge_distances_reduced = space1.get_edge_distances_reduced().to(device), 
                encoder_interpolation_matrix = encoder_sparse_interpolation_1.to(device),
                decoder_interpolation_matrix = decoder_sparse_interpolation_1.to(device),
                point_id_index_reduced = space1.get_point_id_index_reduced().to(device),
                edge_indices_reduced_1 = space2.get_indices_reduced().to(device), 
                edge_distances_reduced_1 = space2.get_edge_distances_reduced().to(device),
                encoder_interpolation_matrix_1 = encoder_sparse_interpolation_2.to(device),
                decoder_interpolation_matrix_1 = decoder_sparse_interpolation_2.to(device),
                point_id_index_reduced_1 = space2.get_point_id_index_reduced().to(device),
                n_units_input = n_units_input,
                n_units_output = n_units_output,
                n_layers_orig = layers_orig,
                n_units_orig = units_orig,
                n_layers_pool_1 = layers_pool_1,
                n_units_pool_1 = units_pool_1,
                n_layers_pool_2 = layers_pool_2,
                n_units_pool_2 = units_pool_2
).to(device)

In [ ]:
# Execute training with best hyperparameters

def count_parameters(model):
    """
    Counts the number of trainable parameters in a PyTorch model.

    Args:
        model (torch.nn.Module): The model to count parameters for.

    Returns:
        int: Number of trainable parameters.
    """
    return sum(p.numel() for p in model.parameters() if p.requires_grad)  # Sum all trainable parameters

print('\n')
print('Compiling GCN optimized model:')

train_losses = []  # List to store training losses per epoch
val_losses = []    # List to store validation losses per epoch

print(model)  # Print model architecture

if Execute_Train_flag:  # Check if training should be executed

    num_params = count_parameters(model)  # Count trainable parameters
    print(f"Number of trainable parameters in the model: {num_params} \n")

    n_epochs = 2000  # Number of training epochs
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)  # Adam optimizer
    mse_loss = nn.MSELoss()  # Mean Squared Error loss

    # Learning rate scheduler to reduce LR every 30 epochs by a factor of 0.9
    scheduler = StepLR(optimizer, step_size=30, gamma=0.9)

    early_stopping_patience = 100  # Number of epochs to wait for improvement before stopping
    best_val_loss = float("inf")  # Initialize best validation loss
    epochs_without_improvement = 0  # Counter for early stopping
    lambda_factor = 0.01  # Weight for the error moment term

    for epoch in range(n_epochs):
        model.train()  # Set model to training mode
        total_loss = 0  # Accumulate training loss

        for data in train_loader:  # Iterate over training batches
            optimizer.zero_grad()  # Zero gradients

            for i in range(len(data)):  # Iterate over items in batch
                x, y = data[i].x, data[i].y  # Get input and target
                out = model(x)  # Forward pass
                loss = mse_loss(out, y)  # Compute MSE loss
                error_moment = integral_load(x, y, out)  # Compute error moment
                combined_loss = loss + lambda_factor * error_moment  # Combine losses
                combined_loss.backward()  # Backpropagation
                optimizer.step()  # Update weights
                total_loss += combined_loss.item()  # Accumulate loss

        average_train_loss = total_loss / len(train_loader)  # Average training loss

        # Validation phase
        model.eval()  # Set model to evaluation mode
        total_val_loss = 0  # Accumulate validation loss
        with torch.no_grad():  # No gradient computation
            for data in val_loader:  # Iterate over validation batches
                for i in range(len(data)):  # Iterate over items in batch
                    x, y = data[i].x, data[i].y  # Get input and target
                    out = model(x)  # Forward pass
                    val_loss = mse_loss(out, y)  # Compute MSE loss
                    error_moment_val = integral_load(x, y, out)  # Compute error moment
                    combined_loss_val = val_loss + lambda_factor * error_moment_val  # Combine losses
                    total_val_loss += combined_loss_val.item()  # Accumulate loss

        average_val_loss = total_val_loss / len(val_loader)  # Average validation loss

        print(f"Epoch {epoch + 1}, Train Loss: {average_train_loss}, Val Loss: {average_val_loss}")

        train_losses.append(average_train_loss)  # Store training loss
        val_losses.append(average_val_loss)      # Store validation loss

        current_lr = optimizer.param_groups[0]['lr']  # Get current learning rate
        if current_lr > 0.00005: 
            scheduler.step()  # Step the scheduler if LR above threshold

        # Early stopping check
        if average_val_loss < best_val_loss:
            best_val_loss = average_val_loss  # Update best validation loss
            epochs_without_improvement = 0    # Reset counter
            model_save_path = os.path.join(model_dir, model_filename)  # Path to save model

            if os.path.exists(model_save_path):  # Remove old model if exists
                os.remove(model_save_path)

            torch.save(model.state_dict(), model_save_path)  # Save model state
        else:
            epochs_without_improvement += 1  # Increment counter

        # Stop training if no improvement for specified patience
        if epochs_without_improvement >= early_stopping_patience:
            print(f"Early stopping at {epoch + 1} after {early_stopping_patience} epochs without improvement.")
            break

    # Plot the training and validation loss over epochs
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, len(train_losses) + 1), train_losses, label='Train Loss')  # Plot training loss
    plt.plot(range(1, len(val_losses) + 1), val_losses, label='Validation Loss')  # Plot validation loss
    plt.xlabel('Epoch')
    plt.ylabel('Loss (MSE)')
    plt.ylim(0, 0.1)
    plt.legend()
    plt.title('Training and Validation Loss vs Epochs')
    plt.grid(True)
    plt.show()

In [ ]:
# Make predictions using the trained model if transfer learning is enabled
if Transfer_Learning_flag:
    # Load the best saved model weights
    model.load_state_dict(torch.load(model_dir + model_filename))  # Load model parameters
    model.eval()  # Set model to evaluation mode

    predictions = []  # List to store predictions

    # Disable gradient computation for inference
    with torch.no_grad():
        for data in test_loader:  # Iterate over test data batches
            for i in range(len(data)):  # Iterate over each graph in the batch
                out = model(data[i].x)  # Forward pass to get prediction
                predictions.append(out)  # Store prediction

    # Concatenate all predictions into a single tensor
    predictions = torch.cat(predictions, dim=0)  # Shape: (num_samples * num_points, num_variables)

    # Reshape predictions to match the original test set shape
    samples, points, variables = Y_test.shape  # Get shape of ground truth
    predictions = predictions.cpu().numpy()  # Move predictions to CPU and convert to numpy
    predictions = predictions.reshape(samples, points, variables)  # Reshape to (samples, points, variables)

    # Inverse transform X_test to original scale
    samples, points, variables = X_test.shape  # Get shape of X_test
    X_test_flat = X_test.reshape(-1, variables)  # Flatten X_test for scaler
    X_test_flat = scaler.inverse_transform(X_test_flat)  # Inverse transform
    X_test = X_test_flat.reshape(samples, points, variables)  # Reshape back

    # Inverse transform predictions to original scale
    samples, points, variables = predictions.shape  # Get shape of predictions
    predictions_flat = predictions.reshape(-1, variables)  # Flatten predictions
    predictions_flat = scalery.inverse_transform(predictions_flat)  # Inverse transform
    predictions = predictions_flat.reshape(samples, points, variables)  # Reshape back

    # Inverse transform Y_test to original scale
    samples, points, variables = Y_test.shape  # Get shape of Y_test
    Y_test_flat = Y_test.reshape(-1, variables)  # Flatten Y_test
    Y_test_flat = scalery.inverse_transform(Y_test_flat)  # Inverse transform
    Y_test = Y_test_flat.reshape(samples, points, variables)  # Reshape back